# 08. Análisis en frecuencia: la DFT y la FFT

**Objetivos de aprendizaje.** Al terminar podrás:

1. Explicar qué mide la DFT: **cuánto se parece la señal a cada senoidal**.
2. Calcular una DFT pequeña **a mano** y con un **doble ciclo**, y comprobar que la FFT da lo mismo (pero mucho más rápido).
3. Convertir el índice $k$ de la DFT a **Hz**: $F_k = k\,f_s/N$.
4. Interpretar el espectro: mitad "espejo", amplitudes, **resolución** $\Delta f = f_s/N$ y **fuga espectral**.
5. Encontrar las frecuencias presentes en señales reales (audio, ECG).

**Requisito:** notebooks 01, 06 y 07.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. La idea: toda señal es una suma de senoidales

Así como toda señal es una suma de **impulsos** (notebook 02), también es una suma de **senoidales** de distintas
frecuencias. El **espectro** nos dice **qué frecuencias** tiene una señal y **con qué amplitud**.

In [ ]:
fs = 1000
n, t, s1 = pds.senoidal_digital(50, fs, 0.2, A=1.0)
n, t, s2 = pds.senoidal_digital(120, fs, 0.2, A=0.5)
n, t, s3 = pds.senoidal_digital(300, fs, 0.2, A=0.25)
x = s1 + s2 + s3

figura, ejes = plt.subplots(2, 1, figsize=(10, 6))
pds.graficar_continua(t, x, titulo="En el tiempo: x = 1·cos(50 Hz) + 0.5·cos(120 Hz) + 0.25·cos(300 Hz)", ax=ejes[0])
f, mag = pds.espectro(x, fs)
pds.graficar_espectro(f, mag, titulo="En frecuencia: tres barras, una por cada senoidal", ax=ejes[1])
plt.tight_layout()
plt.show()

## 2. La DFT (Transformada Discreta de Fourier)

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j 2\pi k n / N}, \qquad k = 0, 1, \dots, N-1$$

**¿Cómo leerla?** Por la fórmula de Euler, $e^{-j\theta} = \cos\theta - j\sin\theta$. Entonces:

$$X[k] = \underbrace{\sum_n x[n]\cos\left(2\pi \tfrac{k}{N} n\right)}_{\text{parecido con un coseno de } f = k/N} \;-\; j\underbrace{\sum_n x[n]\sin\left(2\pi \tfrac{k}{N} n\right)}_{\text{parecido con un seno de } f = k/N}$$

Es decir, es una **correlación** (notebook 07) de la señal con un coseno y un seno de frecuencia digital $f = k/N$.
Si $x$ contiene esa frecuencia, $|X[k]|$ es grande.

### 2.1 A mano: $x = \{1, 2, 3, 4\}$, $N = 4$

Con $N = 4$: $e^{-j2\pi kn/4} = (-j)^{kn}$, que sólo toma los valores $1, -j, -1, j$.

| | $n=0$ | $n=1$ | $n=2$ | $n=3$ | $X[k]$ |
|---|---|---|---|---|---|
| $k=0$ | $1\cdot1$ | $2\cdot1$ | $3\cdot1$ | $4\cdot1$ | $10$ |
| $k=1$ | $1\cdot1$ | $2\cdot(-j)$ | $3\cdot(-1)$ | $4\cdot(j)$ | $-2+2j$ |
| $k=2$ | $1\cdot1$ | $2\cdot(-1)$ | $3\cdot1$ | $4\cdot(-1)$ | $-2$ |
| $k=3$ | $1\cdot1$ | $2\cdot(j)$ | $3\cdot(-1)$ | $4\cdot(-j)$ | $-2-2j$ |

### 2.2 Con código: el doble ciclo

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0])
N = len(x)
X = np.zeros(N, dtype=complex)          # la DFT es COMPLEJA

for k in range(N):                      # para cada frecuencia k
    suma = 0.0 + 0.0j
    for n in range(N):                  # recorremos la señal
        suma = suma + x[n] * np.exp(-2j * np.pi * k * n / N)
    X[k] = suma
    print(f"X[{k}] = {np.round(X[k], 6)}")

In [ ]:
import inspect
print(inspect.getsource(pds.dft))

### 2.3 La FFT: el mismo resultado, mucho más rápido

La **FFT** (*Fast Fourier Transform*) es un **algoritmo** que calcula **exactamente la misma DFT** con $N\log_2 N$
operaciones en lugar de $N^2$. Para $N = 1\,000\,000$: $2\times10^7$ vs $10^{12}$ operaciones.

In [ ]:
X_fft = np.fft.fft(x)
pds.comprobar_iguales(X, X_fft, "nuestra DFT y np.fft.fft dan lo mismo")

In [ ]:
import time
x_larga = np.random.randn(1024)

inicio = time.perf_counter()
X1 = pds.dft(x_larga)
tiempo_dft = time.perf_counter() - inicio

inicio = time.perf_counter()
X2 = np.fft.fft(x_larga)
tiempo_fft = time.perf_counter() - inicio

pds.comprobar_iguales(X1, X2, "mismo resultado con N = 1024", tolerancia=1e-6)
print(f"DFT con ciclos: {tiempo_dft:.3f} s    FFT: {tiempo_fft*1000:.3f} ms   ->  ~{tiempo_dft/tiempo_fft:.0f} veces más rápida")

## 3. ¿Qué frecuencia (en Hz) corresponde a cada $k$?

$X[k]$ corresponde a la frecuencia digital $f = k/N$. Como $F = f \cdot f_s$:

$$F_k = k\,\frac{f_s}{N} \quad\text{Hz}, \qquad \Delta f = \frac{f_s}{N} \text{ (separación entre dos } k \text{ consecutivas)}$$

### Ejemplo: senoidal de 50 Hz, $f_s = 1000$ Hz, $N = 1000$

In [ ]:
fs = 1000
N = 1000
n, t, x = pds.senoidal_digital(50, fs, N / fs, A=2.0)

X = np.fft.fft(x)
magnitud = np.abs(X)
k = np.arange(0, N)
F = pds.eje_frecuencias(N, fs)

figura, ejes = plt.subplots(2, 1, figsize=(10, 6))
pds.graficar_espectro(k, magnitud, titulo="|X[k]| contra el índice k", etiqueta_x="k", ax=ejes[0])
pds.graficar_espectro(F, magnitud, titulo="|X[k]| contra la frecuencia F = k·fs/N", ax=ejes[1])
plt.tight_layout()
plt.show()

print("Los picos están en k =", k[magnitud > 100])

**Dos preguntas:**

1. **¿Por qué hay un pico en $k = 950$?** La DFT de una señal real es **simétrica**: $|X[N-k]| = |X[k]|$.
   La frecuencia $950$ Hz equivale a $950 - 1000 = -50$ Hz: es la **frecuencia negativa** del coseno
   ($\cos\theta = \frac{e^{j\theta} + e^{-j\theta}}{2}$). Por eso **sólo miramos de $0$ a $f_s/2$**.
2. **¿Por qué el pico mide 1000 y no 2?** La DFT **suma** $N$ productos. Dividiendo entre $N$ y multiplicando por 2
   (por la mitad que tiramos) obtenemos la **amplitud**: $1000 / 1000 \times 2 = 2$.

La función `pds.espectro` hace todo esto:

In [ ]:
print(inspect.getsource(pds.espectro))

In [ ]:
f, mag = pds.espectro(x, fs)
pds.graficar_espectro(f, mag, titulo="Espectro de un lado: amplitud 2 en 50 Hz")
plt.show()
print("Frecuencia del pico:", f[np.argmax(mag)], "Hz,  amplitud:", np.max(mag))

## 4. Resolución en frecuencia: $\Delta f = f_s / N$

Sólo "vemos" las frecuencias $0, \Delta f, 2\Delta f, \dots$ Para distinguir dos frecuencias cercanas necesitamos
$\Delta f$ pequeña → **muchas muestras** → **mucho tiempo de señal** ($N/f_s = 1/\Delta f$ segundos).

In [ ]:
fs = 1000
figura, ejes = plt.subplots(1, 2, figsize=(13, 3.5))
for eje, N in zip(ejes, [100, 1000]):
    n, t, a = pds.senoidal_digital(50, fs, N / fs)
    n, t, b = pds.senoidal_digital(55, fs, N / fs)
    f, mag = pds.espectro(a + b, fs)
    pds.graficar_espectro(f, mag, titulo=f"50 Hz + 55 Hz, N = {N}: Δf = {fs/N} Hz", ax=eje, discreto=True)
    eje.set_xlim(30, 75)
plt.tight_layout()
plt.show()

## 5. Fuga espectral (*leakage*) y ventanas

Si la frecuencia **no cae exactamente** en un múltiplo de $\Delta f$ (o si la señal no contiene un número entero de ciclos),
la energía se "fuga" a las frecuencias vecinas. Multiplicar por una **ventana** suave (Hann) reduce la fuga.

In [ ]:
fs, N = 1000, 200                       # Δf = 5 Hz
n, t, exacta = pds.senoidal_digital(50, fs, N / fs)       # 50 = 10·Δf  (cae en un bin)
n, t, fuga = pds.senoidal_digital(52.5, fs, N / fs)       # 52.5 = 10.5·Δf (cae entre dos)
ventana = np.hanning(N)

figura, ejes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
datos = [(exacta, "50 Hz: cae justo en un bin"), (fuga, "52.5 Hz: se fuga"),
         (fuga * ventana, "52.5 Hz con ventana de Hann")]
for eje, (senal, titulo) in zip(ejes, datos):
    f, mag = pds.espectro(senal, fs)
    pds.graficar_espectro(f, mag, titulo=titulo, ax=eje, discreto=True)
    eje.set_xlim(0, 120)
plt.tight_layout()
plt.show()

## 6. Señales reales

### ECG: ¿se ve la interferencia de 60 Hz?

In [ ]:
fs_ecg = 360
t_ecg, ecg = pds.cargar_senal_texto("../datos/ecg_sintetico.csv", fs=fs_ecg, columna=1)
f, mag = pds.espectro(ecg, fs_ecg)

figura, ejes = plt.subplots(2, 1, figsize=(10, 6))
pds.graficar_espectro(f, mag, titulo="Espectro del ECG", ax=ejes[0])
pds.graficar_espectro(f, mag, titulo="Acercamiento 0-5 Hz: la deriva (0.3 Hz) y los armónicos del latido (1.25 Hz)", ax=ejes[1])
ejes[1].set_xlim(0, 5)
plt.tight_layout()
plt.show()

- **60 Hz**: la interferencia de la red, que aparece como un pico aislado.
- **1.25 Hz y sus múltiplos**: el latido (75 lpm = 1.25 latidos/s) y sus armónicos (el ECG es periódico pero no senoidal).
- **0.3 Hz**: la deriva de la línea base.

### Nyquist en el dominio de la frecuencia
Una senoidal de 700 Hz muestreada a 1000 Hz (Nyquist = 500 Hz) aparece en $1000 - 700 = 300$ Hz:

In [ ]:
n, t, x = pds.senoidal_digital(700, 1000, 1.0)
f, mag = pds.espectro(x, 1000)
pds.graficar_espectro(f, mag, titulo="700 Hz muestreada a 1000 Hz: aparece en 300 Hz")
plt.show()

## Resumen

| Concepto | Fórmula / idea |
|---|---|
| DFT | $X[k] = \sum_n x[n] e^{-j2\pi kn/N}$: correlación con senoidales |
| FFT | algoritmo rápido para la DFT ($N\log N$) |
| Frecuencia de $k$ | $F_k = k f_s / N$ |
| Resolución | $\Delta f = f_s / N$ (más tiempo → más resolución) |
| Simetría | señales reales: $|X[N-k]| = |X[k]|$ → sólo miramos $0..f_s/2$ |
| Amplitud | $2|X[k]|/N$ |
| Fuga | frecuencia entre bins → se reparte; se reduce con ventanas |

## Comprueba tu comprensión

1. $f_s = 8000$ Hz, $N = 1024$. ¿A qué frecuencia corresponde $k = 128$? ¿Cuánto vale $\Delta f$?
2. ¿Cuántas muestras necesitas para tener una resolución de 1 Hz con $f_s = 44100$?
3. ¿Por qué el espectro de una señal real sólo se grafica hasta $f_s/2$?

<details><summary>Ver respuestas</summary>

1. $F = 128 \times 8000/1024 = 1000$ Hz; $\Delta f = 7.8125$ Hz.
2. $N = 44100$ (1 segundo de señal).
3. Porque la otra mitad es el espejo (frecuencias negativas) y no aporta información nueva.
</details>